# 01. Connect to the Exchange and Inventory Available Data

This notebook enumerates every study, patient, consent scope, and observation type visible to the logged-in account, then writes an aggregate snapshot to `outputs/inventory_snapshot.csv`. The snapshot contains counts only and is safe to commit.

Run it on the JupyterHealth Hub, where `JHE_URL` and `JHE_TOKEN` are already set.

In [1]:
import pandas as pd
from jupyterhealth_client import JupyterHealthClient, Code

client = JupyterHealthClient()
client.get_user()

{'id': 10080,
 'email': 'elvis.han@ucsf.edu',
 'firstName': 'NONE',
 'lastName': 'NONE',
 'patient': None,
 'userType': 'practitioner',
 'isSuperuser': False,
 'settings': {'currentStudyId': 30004,
  'currentFhirResource': 'AllergyIntolerance',
  'currentOrganizationId': 20001}}

## Studies

In [2]:
studies = pd.DataFrame([
    {"study_id": s["id"], "study_name": s["name"], "organization": s["organization"]["name"]}
    for s in client.list_studies()
])
studies

,study_id,study_name,organization
0,30003,Cardiology Div Study on BGL,Cardiology Division
1,30006,CGM & Wearables Demo,Planetary Research Institute
2,30005,Cosmic Cardio Lab Study on O2,Cosmic Cardio Lab
3,30001,Lifespan Study on BP & HR,Lifespan Lab
4,30002,Lifespan Study on Sleep & BP,Lifespan Lab
5,30004,Nep Pulse Lab Study on BT,Neptunian Pulse Lab


In [3]:
# 1. Full study metadata: every field the Exchange stores, not just the name
raw_studies = list(client.list_studies())
print(raw_studies[0].keys())          # see which fields exist (description, icon, organization, ...)
pd.json_normalize(raw_studies)

dict_keys(['id', 'name', 'description', 'organization', 'iconUrl'])


,id,name,description,iconUrl,organization.id,organization.name,organization.type
0,30003,Cardiology Div Study on BGL,Blood Glucose,None,20006,Cardiology Division,dept
1,30006,CGM & Wearables Demo,Synthetic CGM + Oura wearable demo data.,None,20001,Planetary Research Institute,edu
2,30005,Cosmic Cardio Lab Study on O2,Oxygen Saturation,None,20008,Cosmic Cardio Lab,laboratory
3,30001,Lifespan Study on BP & HR,Blood Pressure & Heart Rate,None,20003,Lifespan Lab,laboratory
4,30002,Lifespan Study on Sleep & BP,Sleep & Blood Pressure,None,20003,Lifespan Lab,laboratory
5,30004,Nep Pulse Lab Study on BT,Body Temperature,None,20007,Neptunian Pulse Lab,laboratory


In [4]:
# 2. Column schema per observation type: what each measurement actually records
def schema(study_id, code):
    df = client.list_observations_df(study_id=study_id, code=code, limit=5)
    return [c for c in df.columns if not c.startswith(("resource_", "meta_"))]

schema(30006, Code.BLOOD_GLUCOSE)     # swap in any Code member

['code',
 'id',
 'status',
 'code_coding_0_system',
 'code_coding_0_code',
 'code_coding_0_display',
 'subject_reference',
 'subject_identifier_system',
 'subject_identifier_value',
 'resourceType',
 'uuid',
 'modality',
 'schema_id_name',
 'schema_id_version',
 'schema_id_namespace',
 'acquisition_provenance_source_name',
 'source_creation_date_time',
 'blood_glucose_unit',
 'blood_glucose_value',
 'effective_time_frame_date_time',
 'source_creation_date_time_local',
 'effective_time_frame_date_time_local']

In [5]:
# 3. Value distributions: units, plausible ranges, and whether the data looks synthetic
df = client.list_observations_df(study_id=30006, code=Code.BLOOD_GLUCOSE, limit=None)
value_cols = [c for c in df.columns if c.endswith(("_value", "_unit"))]
df[value_cols].describe(include="all")

,subject_identifier_value,blood_glucose_unit,blood_glucose_value
count,26900,26900,26900.000000
unique,20,1,NaN
top,10031,mg/dL,NaN
freq,1345,26900,NaN
mean,NaN,NaN,118.146320
std,NaN,NaN,21.388615
min,NaN,NaN,75.000000
25%,NaN,NaN,104.000000
50%,NaN,NaN,112.000000
75%,NaN,NaN,124.000000


In [6]:
# 4. Per-patient coverage: how many days each patient reports, the basis for any dropout model
tcol = find_time_col(df)
df["day"] = pd.to_datetime(df[tcol], utc=True).dt.date
coverage = (df.groupby("subject_reference")
              .agg(n_obs=("day", "size"), n_days=("day", "nunique"),
                   first=("day", "min"), last=("day", "max")))
coverage.describe()

NameError: name 'find_time_col' is not defined

## Patients and consented scopes

One row per patient, study, and consented measurement type. Consent is granted per study and per scope, so this table shows exactly which data each study is permitted to see.

In [ ]:
rows = []
for s in studies.itertuples():
    for p in client.list_patients(study_id=s.study_id):
        consents = client.get_patient_consents(p["id"])
        for study in consents["studies"]:
            if study["id"] != s.study_id:
                continue
            for scope in study["scopeConsents"]:
                rows.append({
                    "study_id": s.study_id,
                    "patient_id": p["id"],
                    "scope": scope["code"]["codingCode"],
                    "consented": scope["consented"],
                })

consents = pd.DataFrame(rows)
consents.groupby(["study_id", "scope", "consented"]).patient_id.nunique().unstack(fill_value=0)

## Observation counts by study and type

`limit=None` retrieves every record, which can be slow for large studies. Reduce it if a study is very large.

In [ ]:
summary = []
for s in studies.itertuples():
    for code in Code:
        df = client.list_observations_df(study_id=s.study_id, code=code, limit=None)
        if len(df) == 0:
            continue
        t = df["effective_time_frame_date_time"]
        summary.append({
            "study_id": s.study_id,
            "study_name": s.study_name,
            "code": code.value,
            "n_patients": df["subject_reference"].nunique(),
            "n_observations": len(df),
            "first": t.min().date(),
            "last": t.max().date(),
        })

summary = pd.DataFrame(summary)
summary

In [ ]:
summary.to_csv("../outputs/inventory_snapshot.csv", index=False)
print(summary.to_markdown(index=False))  # paste into docs/02_data-inventory.md

## What else the Exchange exposes

These answer the open questions in `docs/02_data-inventory.md`: which FHIR resource types exist beyond Observation, and which data sources are registered.

In [ ]:
client.list_fhir_resource_types()

In [ ]:
pd.DataFrame(list(client.list_data_sources()))